In [ ]:
import pandas as pd
import pymysql
from pymongo import MongoClient
import psycopg2
import matplotlib.pyplot as plt
import folium
import time as tm
from folium import plugins
import seaborn as sns
from folium import FeatureGroup, LayerControl, Map, Marker
%matplotlib inline
import json
import urllib.request
from math import radians, cos, sin, asin, sqrt
import matplotlib
import numpy as np
import datetime

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py

#db = client.get_database('telematics_db')




def gettime(x):
    try:
        return pd.Timestamp(x).timestamp()
    except:
        return None

In [ ]:
device_history=pd.DataFrame(list(db.deviceHistory.find({"fromTime":{"$lte":(1559347200),'$gte':(1559347200-30*86400)}})))
#device_history=device_history[device_history['suggestion']=='Battery disconnected']


In [ ]:
len(device_history)

In [ ]:
device_history.suggestion.value_counts()

In [ ]:
vehicle_details=pd.read_sql("select sub_type_installation,vehicle_id,model_name,data_source,vehicle_state,live_time,deboarding_time from vehicle_details",con=postgres)

device_models=pd.read_sql("select * from device_models",con=mysql_cn)

our_devices=['GT06N','WETRACK2','WETRACK-LITE','C100','TL1000','TL250','WETRACK800','AW100','WP30L','GRLAIS','AT50','WP30L','FMB920','FMB207','FMB150','FMB001','LC100_SOS','WETRACK2_SOS','AT4','ATLANTA_OBD','BHARAT101','TRANSYNC_AIS_140','GAGAN']                           


our_device_list=[]

for i, row in device_models.iterrows():
    if row['name'] in our_devices:
        our_device_list.append('yes')
    else:
        our_device_list.append('no')
device_models['our_device']=our_device_list

In [ ]:
pings=pd.DataFrame()
sortOrder="asc"
timeCondition="gte"
count=0
for i, row in device_history.iterrows():
        from_tm=int(row['fromTime'])
        to_tm=""
        
        
        #print(url)
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&toTime={}&sortOrder={}&timeCondition={}".format(row['vehicleId'],from_tm,to_tm,sortOrder,timeCondition)
        data=json.load(urllib.request.urlopen(url))
        
        try:
         #   print(data)
            data1=pd.DataFrame(data['data'][str(row['vehicleId'])])
            data1['time_diff']=data1['time']-row['fromTime']
            data1['suggestion']=row['suggestion']
            pings=pd.concat([pings,data1])
            
        except:
            count+=1
                

In [ ]:
pings.drop_duplicates(['usedSat','vehicleId'],inplace=True)

In [ ]:
pings=pd.merge(pings,vehicle_details,left_on='vehicleId',right_on='vehicle_id',how='left')

In [ ]:
pings=pd.merge(pings,device_models,left_on='model_name',right_on='name')